# Vigenere cipher



1. First, I cleaned the ciphertext by keeping only lowercase alphabetic  characters (though in our case its already cleaned), since the Vigenère cypher operates on letters modulo 26.

2. To determine the key length, I computed the Index of Coincidence (IC) for key sizes from 1 to 20 and selected the length whose average IC was closest to English text.

3. Once the key length was known, I split the ciphertext into columns corresponding to each key position and treated each column as a Caesar cipher.

4. For each column, we applied chi-squared frequency analysis against standard English letter frequencies to find the best shift, thereby recovering the key.

In [2]:
import string
from collections import Counter

# 1. Input loading and cleaning

raw_ciphertext = """kbjamkkgfdrwizncqlkitvshycxoslzyxtllyyatjdrajzdmcuufplyizrflyyatjdrajhykrmrljdfhjefwiyblqsecsogyfzfnrgisspyjssfyblyyucmhicjemjrhjyedzmmxyfyuixyfrajorgxyylngjntqdatyjdrssfndfwuizcngjnrlqlvlgpjgeajorgtuqnslkumpdwcnqtiwrzndfglntquskywtllycxccefnjggdculpfajikqguvusojamcsrpgfgbppwzhfoyjbnmlruyyidfwuhtedsizwzksxljplkccrjngexxfpjfosocvfhfwjkzxjdzqrxjyqwxlthrzkbjxcfvguwmqvxnyrzvcsogyfzfnrgismlbfffjtqmiyrzpwfpjcrzvsbppwyuwojquyxtpssfjnmegustmfjztcbwtysedgceszpajuhljulnylzgpuslbwgnnyrzvuwemxrmxzaarnnyeoznmzrzvlxlkgeaxepseajcqzvuuacsimjtrzvlucmmuiwtjdrnjlqwrnfywjrnjefwgixeksjnjcfsuvzejaknqpagdjfywfflmlbzvgznflfxtlrlzgjdfwklnpbzzmmllvrnbcglzhlltwimjzplniysylkbjxmnvgjyrgwnmpjwrpjdyfunmpadfoidmxkbjdiqnywpcffolsrgwcqwjawybtrzaiddsuyjlhcjvnmpqwennxcfkmyzuzzwmsckfolsrlfangcwojwpqkzismslxiivlgnmysylkbjamgizjwjgnqtfjvyuapdwcnneykkbjrgxkikllwnfnqcawmtxcyvhnpmxkbjlpsscfylaxbydfsucszlweclsrknyueyorsyscliyjdjwrpjdyfuuqwyfuljajstyiefwdqnefsduhlbsdcxpbjfuisgvzhlefwtftfbkwltxtavqbtrziibdmxkuqwfglmjdrzvjtdrermyppkjuqlpqnuxdkscfmpfsuntnmgbbndmoegjljknbnnfzvoxpblfmmlpwncyspskusllgijmllyzlqzdlyyatjdrajhfgucizbvaigddgibnxuzvhnyrzvyaplaeaysckdippzwxusemullqfnxiirefwmcqwyyvwthqzvxxllvkbjngurffdazzlupbaeyappqsoxsuzvhyscevhitasenxzdlyygljkvwydyfxnmpgjjbwtjdjisrqaenmpgjuunwwevyytlygffncoyysllqgijeuzfbfoylkyrarwunthyltbyscefpjxcfkikefwcyfgckzhyscvvhxpzsdvtzrzzwpprknizwbzrpjqcdkulsmkkfddfamywcsfuibyfajvfnilyyuzqlduxecjnizwbdzamefajfnerdvffxnsexhljdfoycylrhwlrseqtfjvjcyzsljcipusznnyexflysgktuqwyfucsdrwrxtqagdcsrgfrntyawnizwbjvjqjbaustfascfrpqaiqmlrsiydzsvfcsrrzvjtdrermyppofoqoykbcrfqlsylzgfxntwgyynysccznhscfwcwpuglfimclyyfyqovluryfunmpngjnrlqlvlbzsdumfjmzcyyefwbcynfwezncctvztcyoycqpjaxbyxcepjnacxzlxeylcuxepskushmmcxjyrwiqnefhlzkpbglnhscwbmategiizdjqsfthgfxcsemswffxcsccapagrfyzjaxbyefwkiglaufnmtqofoqoeamyyschfmyxykkywllggjtcrmecyjmxtisgcjjcsruwcfwlrsejjcfsgmmpuglfimcyzhizwglljxcesywllqkbnyegwstfpefnmpplyuyhykrzjcracyxfzbvwycylrhulplcswpkwdvjccvrhiayjkfdogvenmppxrnmppzrxgpcfwisocjfzmpplyusscjdiyscjycrdfwiyhzjdvwypbefljggnzxqjfwlmjorgtirpfgdynyrzvyaplaeafqrwibndugiefybgeytcrofyaplaeaxdrgfxtfrefljnjwrlqjrzrhtefwimqtiwgchesjvmnyfwigjxmjplfeyfnizwbkzntyrzvzqzmjeyfcrzvjtdrermyppkwyjeykdyrzpavmhcmouyitlmgisscjjbjnydcyiemezhiljaknqpzjfnmpplyuydfwyuillvyibzlkfgjmwyfhjnjglxdoyqjbjsyvgffjcvrnktqzzhlhglybnxmfkbjpbyvikefwgisouakbfeuaxztcyerejmcdzyapdajbnyejfxxfazccyejwzhhtbwenxhmmcxicgnvizeejvuyppwmyseqxiirscjdcsorzlmfdrzvsyljcvxneuglfizdlvhlprnvldwylvusorzvjtdrermyppofoqodwvfyzmdrtdemvfusjagfenyeskuqwpskushmmcxyscfyuxegdpfnrflkbjqgjvusorgrmydmevoswcsmyspbtiyfouzzwmhglynmpagcxwpkfrhydmxkbjxmjecsrkwrfblqweizrfxflyscaimzanwiisdmevyaplaeaxdcskyilrzzmipqczhyscuflspphxikefwsclpkhksxscvkbjamkkgfdrwintzuglfinydcouxceflnpqgwbndmoebtxcgwbndkgkbjcyfubndqajnjcmxkbtdcxflbsmezhmtqwocqpfajbjlplnuxdyvdyrzpavmbsguyqjccscqfjqzrosegfxbnxzmkqmtazyyhzsduhterscefmmmkqneflyyrplgwnmpdstntcwlyizrfzvztflvycrdcdwhfesjrfqjpwtuqwgfxnmpkscizogfkbjapwjysncgwnmpqadjqpjaknqpeaiffybkfcynyevugzslkbferzvancjofoqoydcoiprgycxacggfjlqefnmpptiiyscjrhidgkkywlqawmmpfsueszufkbjxydcbjcjawynydstnxsczrxfnmegfjechzwyfpwfzjlazfhjzdlyyrayaenjogfyywwglkfjscsintycffisosjzhllzjvuptllyywlgfjnmppwnuxlagffxzdlsljpxwsfthgfxnmpqevfqzdlyyilkhxlfdqsexqpynvmnyrzvbteqmezjwrdzejefwnuwxzjvuysgfxikefwkcwpbwrlysmffhjdzgusfacjjcxecfkvncbovhyzlscfyscswnjclgfhwpnwrnnyelyygfpvvhtqgljispagdjqlgfkcsyyllljdymucjyawtbfxzwinmpngjnrlqlvlmlbffnmtlykiizrzvmmtkevltqrzvzwpqzcsblqzvxqpynvmfyblyygllcvxzapwdhfyrkfzyscjvnwpylzhlcyaewqzsvjqjcckzameqlfmjpyfunmpngjnrlqlvlblqornhsgfxnmpksexysgfbcsrrgycrdcdwimtdgefddmevenybjvxxzsdnywplwrlofqlfhjwmnzhlsserhgpgfxqmzkatizwbzffiycsigdscsinysgknuxpvstnqjfwnysemfkiysgfbqmlrlyuymgjuqfdrjpcsrrgjudaesexneusjnmpqsdykpcdzhlhfatbyscellrfpaeaqpynvmbppwjnwttaeayzcpgljdqtlnszmfveszukflbzsduvjwgwmyysyljohsyfzxjlkaxbyljkfnfvchfmxpqkziszdsecqwnszxatjdrajamkkgfdrwicsefwuyjaqacysekauxfjgfkywgydfzmtqoflpefwgixeksjnjcqaxbjoyfuwfwjwuizepskuscylrhblqlyysdnjrqqtlysyspylynmpemrpfepwvvzdgdpysryyvxnycskcsrsficupemrpfdylkbjgmatytqfwigfdrwimmppseoumpwrnmwckjfddyqzhlhcjvstfascfnyeevxfoyanuxefaeenyekrciefwgixeksjnjcmxkyfnfaeadzslfljlbsexyscfwiwefwiyxemxkbjldlvlszmfyyylsyynmpplyyfwnzrvjerzlmnyynvlddfginytkwiuyllzrxlzrsjzfcykkbjommsfjnmfjisllljcydcwdyilqlyizrflyyxsmovlxzdlyyxpykfhbzsduhjgcjvhinyfrfxogltbjdyfubtwjgnmbppwrfqztwizqzuaeabtrznuyppvrsfybfzamerzvjferwiikcyaeqfdfwrlillvkbjnpgrenyegwzwzekkbjggdculppgrxxmcurgjtkhrmxlzdvusoksiejegfxbforgsyizlwzhuflljispfwrpnwwucizocvdiwygfxnmpngjnrlqlvlxwglkfjashzfmlbtvyswmfxqftraeatfrkzxjefwuitcdgibjcascfgfrffnmpyjzhltrsjoxfydjbjemgbouscjuilpyjvxgzmcrhidjgnfdpllvljorzvltzkkyykzsfubjcksjnjcqliyynfwuizemfycxmcvrhiefaeenyelyuyscormwpqlzhldfwnuxlzglnyzpwkcwpnyfhytnlfybscfjbjdsvuyswwzvuwofwihfxcjrnfyqzvnzclwuuyzluvusoykbyihcjvstfqdvyutlyuuilrzvjtdrermyppaeuuwyaenngcnfchpqszxnlkffnbpjdwyjwkqyyfogkznappqyiytllyyqzlwccspqkfzmtqwocqpyfucsefwxftzkgwnmppszhxsgkrcqtlysiijlwvxjoydznywclvhippfllxtlyyyqzlyvxyzpwdyrmcjkbjemmtbtyrzvztcczvuizdkfzysyfumbtrzkcsvjaeagcyuvfjeqlfcrleaeyyschiyxpluvikwmnzhlhmerhmzmvkbjycsihjdqgwgtefwiusoqajnjcyfunmpcpzfjhykeiyogkrjuzgfkyicylrhhpykvxyzzwrfnerdvancjkyyfemftyxechgyitllfnmpngjntqkgkbjcascfjogfkbjggdculpbgtntcesmyyschrnnpllycxagdcmferzvjwznwicsecjmuqdqskouljdeclsrtpbndnacfthagfejofajawfcdwiwsgerhiptwisszusexyscfrmppbsiydzsxvyqtlyrfnerdvvjerwixfoyakqfdqgdyytkwsykzpwkbjamkkgfdrwiqnefovupplwuvtowormfmjwkiqpynvbndqategpbffgtccgwnmtqkrciscoznmocuzmnzladoxeewkuycyfjzjcfwrntyawnltecgwzyzascwzersrhfandzwfeggeztcyliusddwiisefwxltflvfzyscmebjljlycspqkfzyschcuhppwccjgcvwltxfwixzegwjuxysjjywlrseullgfkitvshyywzjvgffncglnxtbwkbjomgivzeqzvhtwmfxywscsixysckrgjzjvtuqwqzvqtfjvjirpradyxacwgcsdgvvzzcramyqjrgwcsorzvjtdrermypphxmneraeatyfajwmlgjflxepwkwmpbgebndzwuusoqlrlnyessmjyrezhipbdpcsemlyyftpoycqppskushykrqftraeamppurfqefwgixeksjnjcusjublglzhllpwgfdemzzmfandzwfeggenmpeaifwpyvyywzjvcyxdmfjiappsextgcjraftlzvllccskzjlpormqpqlnbjyrzvwfwjurgjdfwdclsrtvztflvnusegfxcsefwuizmjwtisdmfrhydylcuxeyxkywluwveyscurfqogvtirpmfvyaplaeabtrzrhtgcjwfthgfxbjlpliuylljlmmpbaentefwiitxuakbmppovljjmmtuqwgfxgjoyvrnmpngjnrlqlvlxlgvzurrmaeafhyqkirzpjfqwlrseqmppwrljjmmxinyevrxftyexinyezfgjhfweqnwjqfohzkwsuhvgsdhteagdcsrzstewlrseuxvcveitefwikzpqlzisefwgixeksjnjcmxycxzufrwhzpvnysemfkiypjdyywefskbndyhgfnnylzisqmjrnwllkwywsyvsyjypwayhecvjimpfsuljdgyeyisgkgixeyfuqfdegzhlsmevztcydfhlegevhjtrzvltqrzvgxamcvuszrzvlbzpvkbjwyegqjyrgexnxjqsowygfxusodjfgfwcsbcszlwtiwycjfzysclyuynfornjcbjzjupbkkyfogdpcsemseyfcrzvhapqkvftyrzvzqzmjsyspylycyldlvlfhfacywlrseltdcsexbpllfzkemlyyptruyysemhiyulpwkbjxcscvzeqzvqfdlgkmtaeilchvytfoytrsjiszrzvlilwkdusjlwnnmtlyjntefaeetqfsuysecjvxmppdznywctiunyuzvhyschfmyxykkywsyvwcstqzvxmtqkljupplyyltpdjoiocfcsfdiwubnxbsuubtjdpizeycvgjemqfowsmevnmpngjnrlqlvlqlsyyyihfskustbwrmftbzvvzefwuciymlkbnyiakhjnckjuwjrgvruwyaentefwxcwwuzvljtldrsyscssmzcbaksysylnbtwcfzamegfyywhyczhlllvzhmppviyfxqlyyuzqlduxecjjfffezzhlcchcsmlsfkyiscjnbfeyfzxjlmfxyyegfxoutllyyrzpfzhlefwgixeksjnjcdglhisgksuyspwrxdsczrxxesubntsgktuqnslkumlzakikmylycsrgfnuyppviubyyfuejaraejneazvlxtlkkyfomxkuptlyrjqflyvcsefwicappsjqfdrzvwzdrgdikefwmcqwyyvztcqgdywpykfhtcmlyywefwxcwwaglfiymlrmpsgervtfrlyyytkwfzmtqvvjfcrmiyxzqzvbfodwkwmpblyyblrwizwzklyywttwiftyetvztccklhwtqwkbfegljbtfjvsywpyvpuxpyjcsfdfwdclsrorhytrswnjcrzvvfefurgjlascfkzpjrnfyqzvysecjvxszgkvfjdqdpusojgfejoqacysejqzhyzfwigfdrwimklawwiwzpvvlxefwduxecjjunowglhjpbffngpyfoctfqssizekqxinyesnudcylrhndfscfypjddsxfauvmxzplfftziswnjcwglnmpqwniwoqovljvgfufdxcsenszbglvymslzhxnpmkugwcsiyyscorsxzdsnirllkyyfcrhxlfeyfyuimmjeyrllqrmhzjvzhlqpgdbjcksjnjcuakbtfrufguwyaengfrlyyxpiaexbzpvjmmpaglfiymlsyfcqzvvzcqlfoyhcwgcsryfumftbffhtjmmeyjolgknjwjsesgzbqrhdefaeafeydcugzsldynomfkqfyrlfmylwgebjcclyyuzqlduxecjnuxosesztflvvxmpfsuhjgcjjyjypskuswgcvnmtqtvztcclyyspuaewzxzwenifjqrlwttwuusorzvjtdrermyppzrpnyeyzpjymnvlhsyjxyucchrljorguyulplaoxezwwiwpqlrlytlyyyhljdvxwlrseusoqszxmppwzmxzkwkbnyexfldzsayiupglncqwiwvjdzsxflxzkwccyejwkcrpfwsltfezkizedjfgmtqhfwpprlyybsmdviksgkdisefkjuqlpqiyylgfzhlzldpuycgxcykzpzzmycynvfqtlyvruplkvmyscfiuyllxvfqlrzzmkpclrhinpavxtsbsuunapspstfbgenlttwdyfywlycsrbgennyyfpqfjrjfogwcssizekwrhiefwemmppseublwglntqqaxbyefwgixeksjnjcfwrpjoykzamemgbousgktuwaclsulaslycxfktiyqwygmywsgkjbtfjvvlfybstwtxnsecjozqrgfyasildtlyycxxyfpwtwmmiyiegfklzyizvmqzudpgfocxflysctfuyhfwebjrmlzhfyblyygzylnuxflvvlblwsexyscjrcsdugcfjypamywwgcvuxepwrgtqrwrlxhcdccsrshwltxrzvyfcrzjqncjwuusoqgsvjoylyywmmojnmplzvzjwrsgunyylyyfcrlyylcgwwmycgubysqyuvikltacffrcyzlqdcwdyiemjvjwpqwenkzpzzgyschxawpyllhxamcvhuppnrxnyeyicjqmxdiyscjvuwefzvlxpjxrntyclzgjsczrxfygegoqdclfatmyubusozjzhlluspuqzlyncysfadnmlrdfhjdmevqftdxflxliweikefwniwwbtlnyscozhisyvaoxedacfjorzvmftjkkbjmmskbfoegkqjwjaentefwdciojwfzysclllgfjwenhfpjvhyllvrfwpyvpnmptacffrcormqpdlsymtlvrhitrkfoywwaeagfpfzhlrpglhinyevcsdgyynxzrzvnwltwcfjczgihjzllyygccsjntqrzvmbtdlwfthgfxlngcjtisdmdvxmtkkvfkhglyjmtjgjiusgurfwpddvwytmfjisefweormcjcyxdkwvnnyekrhiayjkcsrqyfcsrmfzhyscoflqomfuyfeflyylccskjfcraeakcmenbnnfffhjcclllsdzmklfeyfyuiymhycqzqggbddfwnuxhyfuywtlyrvtfrlyyuzqlfzktawzhfqjgfxtqrwrlxtrersgprzrnxsczrxxegdcuqfpczhlsmhvcsdmevwtclwiikscjyyfcrlyuyscjuuiluglficclllsllvkbfegknbddfwtizwbffnypyjyywdcdwublwscuxqmjfowqmgccxsfmdusyyllljtrkwisokajnfvckrljacjjcxecfknmpbatnfeckfzwpykfhyliwrftyelzgjemsjmjcrlyyncmoemblwlyyxfpwjnucmgwmrpyfnbnwcsiyitqtvfnptwuzfwqwyiupgktfzyelfqnefscftyckdclsrsexrlgfkcqwyvrshzkwjqmplakbfdqmtejorzvbjlpluldllvznkzpuzvqjzjvupdrziizrfakmgzlvjusobwguweqswnjcrzrnhzkwjnmpkajywjmxrqfvcfzhlllvkbjymftyfryaenmpnycisrgfxntrclsuhvgfkiyscertjzdlyyxlkwdcxeycvmpfvszbtwhn"""

ciphertext = "".join(c for c in raw_ciphertext.lower() if c in string.ascii_lowercase)


# 2. Use IC to find Key len

def IC(text):
    freq = Counter(text)
    N = len(text)
    return sum(f*(f-1) for f in freq.values()) / (N*(N-1))

scores = []
for k in range(1, 21):
    avg_ic = sum(IC(ciphertext[i::k]) for i in range(k)) / k
    scores.append((k, avg_ic))

key_len = max(scores, key=lambda x: x[1])[0]


# 3. We use Chi squared for key recovery

english_freq = {
    'a': 0.08167,'b': 0.01492,'c': 0.02782,'d': 0.04253,'e': 0.12702,
    'f': 0.02228,'g': 0.02015,'h': 0.06094,'i': 0.06966,'j': 0.00153,
    'k': 0.00772,'l': 0.04025,'m': 0.02406,'n': 0.06749,'o': 0.07507,
    'p': 0.01929,'q': 0.00095,'r': 0.05987,'s': 0.06327,'t': 0.09056,
    'u': 0.02758,'v': 0.00978,'w': 0.02360,'x': 0.00150,'y': 0.01974,'z': 0.00074
}

def chi_squared(text):
    N = len(text)
    freq = Counter(text)
    return sum((freq.get(c,0) - english_freq[c]*N)**2/(english_freq[c]*N)
               for c in english_freq)

key = ""
for i in range(key_len):
    col = ciphertext[i::key_len]
    best_shift = min(range(26),
        key=lambda s: chi_squared(''.join(chr((ord(c)-97-s)%26+97) for c in col)))
    key += chr(97 + best_shift)

# 4. decrypt

def decrypt(text, key):
    return ''.join(
        chr((ord(c)-97-(ord(key[i%len(key)])-97))%26+97)
        for i,c in enumerate(text)
    )

plaintext = decrypt(ciphertext, key)

# 5. output print

print("Detected key length:", key_len)
print("Recovered key:", key)
print("\n===== DECRYPTED TEXT (START) =====\n")
print(plaintext[:2000])  # preview
print("\n===== DECRYPTED TEXT (END) =====\n")
print("Last 10 characters:", plaintext[-10:])


Detected key length: 6
Recovered key: ruflys

===== DECRYPTED TEXT (START) =====

thepostmasterfirsttookuphisdutiesinthevillageofulapurthoughthevillagewasasmallonetherewasanindigofactorynearbyandtheproprietoranenglishmanhadmanagedtogetapostofficeestablishedourpostmasterbelongedtocalcuttahefeltlikeafishoutofwaterinthisremotevillagehisofficeandlivingroomwereinadarkthatchedshednotfarfromagreenslimypondsurroundedonallsidesbyadensegrowththemenemployedintheindigofactoryhadnoleisuremoreovertheywerehardlydesirablecompanionsfordecentfolknorisacalcuttaboyanadeptintheartofassociatingwithothersamongstrangersheappearseitherproudorillateaseatanyratethepostmasterhadbutlittlecompanynorhadhemuchtodoattimeshetriedhishandatwritingaverseortwothatthemovementoftheleavesandthecloudsoftheskywereenoughtofilllifewithjoysuchpgwerethesentimentstowhichhesoughttogiveexpressionbutgodknowsthatthepoorfellowwouldhavefeltitasthegiftofanewlifeifsomegenieofthearabiannightshadinonenightsweptawaythetreesleavesandallandrepla